## AR simulations: evaluating direct and indirect curvature estimates
We simulate 100-dimensional AR(1) population trajectories across a range of autoregressive coefficients and measurement-noise levels to characterize the relationship between temporal persistence and trajectory curvature. We then evaluate how accurately the AR(1) coefficient and implied curvature can be recovered from noisy, repeated responses using the direct and indirect estimation procedures. Analyses are run for both 10-run averages and a single-run control. This notebook supports Fig. 1e–g and the corresponding 1-run control figure in Extended Data Fig 1.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().parents[1]
sys.path.append(str(repo_root))

import numpy as np
import statsmodels.api as sm
import os 
from matplotlib import pyplot as plt
%config InlineBackend.figure_format = 'retina'
plt.rcParams["figure.figsize"] = (5, 4)

from src.AR_sim_core import (
   make_signal_innov,
   make_noiseless_arp,
   make_additive_noisy_arp,
   zs
)
from src.fmri_core import (
   avg_pop_curvature
)

#### Simulate noiseless and noisy AR(1) population trajectorues 
Generate 100-dimensional AR(1) population trajectories across a range of autoregressive coefficients and measurement-noise levels. Ten repeated responses are simulated and averaged for each noisy condition, and direct trajectory curvature is computed for each of 500 simulation iterations.

In [ ]:
# Main analysis: nRuns = 10
# Extended Data Fig.1:  1-run results: set nRuns = 1 and rerun this section (beta_eff = 1.0 because
# run-varying shared noise cannot be separated from signal with a single run)
nRuns = 10

ar_coeffs = np.linspace(0.1, 1, 10)
nBoots = 500
nT = 241
n_voxs = 100
corr = 0.1                          # fraction of innovation variance shared across voxels
beta = 0.7                          # fraction of shared variance assigned to shared signal
signal_innov_std = 1.0              # latent signal (total innovation standard deviation)
added_meas_noise_std_high = 3.0     # additional measurement noise
added_meas_noise_std_low = 1.5

noiseless_timeseries = np.zeros((nBoots, len(ar_coeffs), n_voxs, nT))
noiseless_est_curv = np.zeros((nBoots, len(ar_coeffs))) 

noisy_timeseries = np.zeros((nBoots, len(ar_coeffs), nRuns, n_voxs, nT)) 
avg_noisy_timeseries = np.zeros((nBoots, len(ar_coeffs), n_voxs, nT)) 
noisy_est_curv = np.zeros((nBoots, len(ar_coeffs))) 

high_noisy_timeseries = np.zeros((nBoots, len(ar_coeffs), nRuns, n_voxs, nT)) 
high_noisy_est_curv = np.zeros((nBoots, len(ar_coeffs))) 
avg_high_noisy_timeseries = np.zeros((nBoots, len(ar_coeffs), n_voxs, nT)) 


for boot in range(nBoots):
    for a in range(len(ar_coeffs)):

        if nRuns == 1:
                beta_eff = 1.0

        else:
                beta_eff = beta
                
        # Noiseless (latent)
        _, shared_signal_innov_noiseless, _ = make_signal_innov(
            n_voxs=n_voxs,   
            nT=nT,
            signal_innov_std=signal_innov_std,
            corr=corr,
            beta=beta_eff,
        )
        private_signal_innov = np.random.randn(n_voxs, nT) * signal_innov_std * (1-corr) # use the same private signal for all cases
        latent_ts = make_noiseless_arp(ar_coeffs[a], nT=nT, n_voxs=n_voxs, private_signal_innov=private_signal_innov, shared_signal_innov=shared_signal_innov_noiseless)
        noiseless_timeseries[boot,a,:,:] = latent_ts 
      
        # Noisy 
        _, shared_signal_innov_noisy, _ = make_signal_innov(
            n_voxs=n_voxs,   
            nT=nT,
            signal_innov_std=signal_innov_std,
            corr=corr,
            beta=beta_eff,
        )      

        for run in range(nRuns):
            # for each run, generate shared noise (shared across voxels, not runs)
            _, _, shared_noise_innov = make_signal_innov(
                n_voxs=n_voxs,
                nT=nT,
                signal_innov_std=signal_innov_std,
                corr=corr,
                beta=beta_eff,
            )
            # combine all the sources for innov term
            signal_innov = private_signal_innov + shared_signal_innov_noisy[None, :] + shared_noise_innov[None, :]
            
            # add low measurement noise
            noisy_ts = make_additive_noisy_arp(
                ar_coeff=ar_coeffs[a],
                nT=nT,
                n_voxs=n_voxs,
                signal_innov=signal_innov,
                measurement_noise_std=added_meas_noise_std_low,
            )
            noisy_timeseries[boot,a,run,:,:] = noisy_ts
            
            # add high measurement noise
            high_noisy_ts = make_additive_noisy_arp(
                ar_coeff=ar_coeffs[a],
                nT=nT,
                n_voxs=n_voxs,
                signal_innov=signal_innov,
                measurement_noise_std=added_meas_noise_std_high,
            )
            high_noisy_timeseries[boot,a,run,:,:] = high_noisy_ts

        avg_noiseless_ts = latent_ts
        
        # run averages
        avg_noisy_ts = np.mean(noisy_timeseries[boot,a],axis=0) 
        avg_noisy_timeseries[boot,a,:,:] = avg_noisy_ts
        avg_high_noisy_ts = np.mean(high_noisy_timeseries[boot,a],axis=0)
        avg_high_noisy_timeseries[boot,a,:,:] = avg_high_noisy_ts

        # compute curvatures
        noiseless_est_curv[boot, a] = avg_pop_curvature(avg_noiseless_ts)
        noisy_est_curv[boot, a] = avg_pop_curvature(avg_noisy_ts)
        high_noisy_est_curv[boot, a] = avg_pop_curvature(avg_high_noisy_ts)


#### Relationship between AR(1) timescale and curvature under noise (Fig 1e) 
This shows how direct trajectory curvature vaires with response timescale under noiseless, low-noise, and high-noise conditions. Points show the mean across simulation iterations and error bars show the 95% CI. 

In [ ]:
# Compute mean curvature and central 95% intervals across simulations
noiseless_mean_curv = noiseless_est_curv.mean(axis=0)  # average across bootstraps           
noiseless_lower = np.percentile(noiseless_est_curv, 2.5, axis=0)   
noiseless_upper = np.percentile(noiseless_est_curv, 97.5, axis=0) # shape: len(ar_coeffs)

noisy_mean_curv = noisy_est_curv.mean(axis=0)             
noisy_lower = np.percentile(noisy_est_curv, 2.5, axis=0)   
noisy_upper = np.percentile(noisy_est_curv, 97.5, axis=0) # shape: len(ar_coeffs)

high_noisy_mean_curv = high_noisy_est_curv.mean(axis=0)             
high_noisy_lower = np.percentile(high_noisy_est_curv, 2.5, axis=0)   
high_noisy_upper = np.percentile(high_noisy_est_curv, 97.5, axis=0) # shape: len(ar_coeffs)

# Compute error bars: distance from mean to lower/upper CI
noiseless_yerr = np.vstack([noiseless_mean_curv - noiseless_lower, noiseless_upper - noiseless_mean_curv])
noisy_yerr = np.vstack([noisy_mean_curv - noisy_lower, noisy_upper - noisy_mean_curv])
high_noisy_yerr = np.vstack([high_noisy_mean_curv - high_noisy_lower, high_noisy_upper - high_noisy_mean_curv])

# Plot with error bars
plt.errorbar(ar_coeffs, noiseless_mean_curv, yerr=noiseless_yerr, fmt='o', capsize=3, label='Noiseless',alpha = 0.5)
plt.errorbar(ar_coeffs, noisy_mean_curv, yerr=noisy_yerr, fmt='o', capsize=3, label='Low noise', alpha = 0.5)
plt.errorbar(ar_coeffs, high_noisy_mean_curv, yerr=high_noisy_yerr, fmt='o', capsize=3, label='High noise', alpha = 0.5)

plt.xlim(0, 1.1)
plt.ylim(90, 120)
plt.xlabel("AR(1) coeff")
plt.ylabel("Curvature (degrees)")
plt.title(f"AR(1) coefficient vs curvature ({nRuns} run{'s' if nRuns != 1 else ''})")
plt.legend()
plt.show()

#### Recovery of AR(1) coeff from simulated population responses (Fig 1f)
Estimate the AR(1) coefficient from voxel-averaged simulated responses and compare the recovered coefficient with the ground-truth value. Estimates are shown for noiseless and noisy (high noise) conditions. Points show the mean across simulation iterations, and error bars show the central 95% of recovered coefficients.

In [ ]:
# average across runs for noisy data 
avg_noisy_timeseries = noisy_timeseries.mean(axis=2)   # (nBoots, n_alphas, n_voxs, nT)
rec_noiseless_ar_coeff  = np.zeros((nBoots, len(ar_coeffs)))
rec_noisy_ar_coeff      = np.zeros((nBoots, len(ar_coeffs)))
rec_noiseless_est_curv   = np.zeros((nBoots, len(ar_coeffs)))
rec_noisy_est_curv       = np.zeros((nBoots, len(ar_coeffs)))

for boot in range(nBoots):
    for a_idx in range(len(ar_coeffs)):
        # --- noiseless ---
        mean_noiseless_ts = noiseless_timeseries[boot, a_idx].mean(axis=0)  # (nT,)
        zs_noiseless_ts   = zs(mean_noiseless_ts)
        noiseless_model   = sm.tsa.AutoReg(zs_noiseless_ts, lags=1).fit()
        a_noiseless       = noiseless_model.params[1]  # AR(1) coefficient
        rec_noiseless_ar_coeff[boot, a_idx] = a_noiseless
        # get curvature from formula
        theta = np.clip((a_noiseless - 1)/2, -1,1)
        curv_noiseless = np.degrees(np.arccos(theta))
        rec_noiseless_est_curv[boot, a_idx] = curv_noiseless

        # --- noisy ---
        mean_noisy_ts = avg_high_noisy_timeseries[boot, a_idx].mean(axis=0)  # (nT,)
        zs_noisy_ts   = zs(mean_noisy_ts)
        noisy_model   = sm.tsa.AutoReg(zs_noisy_ts, lags=1).fit()
        a_noisy       = noisy_model.params[1]
        rec_noisy_ar_coeff[boot, a_idx] = a_noisy
        # get curvature from formula
        theta = np.clip((a_noisy - 1)/2, -1,1)
        curv_noisy = np.degrees(np.arccos(theta))
        rec_noisy_est_curv[boot, a_idx] = curv_noisy

# Compute means across boots
noiseless_mean_ar = np.mean(rec_noiseless_ar_coeff, axis=0)   # shape: (len(ar_coeffs),)
noisy_mean_ar     = np.mean(rec_noisy_ar_coeff, axis=0)

# Compute 95% bootstrap percentile confidence intervals (asymmetric)
noiseless_lower = np.percentile(rec_noiseless_ar_coeff, 2.5, axis=0)
noiseless_upper = np.percentile(rec_noiseless_ar_coeff, 97.5, axis=0)
noiseless_ci = [noiseless_mean_ar - noiseless_lower,
                noiseless_upper - noiseless_mean_ar]

noisy_lower = np.percentile(rec_noisy_ar_coeff, 2.5, axis=0)
noisy_upper = np.percentile(rec_noisy_ar_coeff, 97.5, axis=0)
noisy_ci = [noisy_mean_ar - noisy_lower,
            noisy_upper - noisy_mean_ar]

# Plot
plt.figure(figsize=(5, 4))
plt.errorbar(ar_coeffs, noiseless_mean_ar, yerr=noiseless_ci,
             fmt='o', label='Noiseless', alpha=1,
             capsize=3, color='gray')
plt.errorbar(ar_coeffs, noisy_mean_ar, yerr=noisy_ci,
             fmt='o', label='Noisy', alpha=0.7,
             capsize=3, color='blue')

# y = x reference line
plt.plot(ar_coeffs, ar_coeffs, 'k--', linewidth=1, alpha=0.5)

plt.xlabel("Ground-truth AR coeff")
plt.ylabel("Estimated AR coeff")
plt.title(
    f"AR coefficient recovery "
    f"({nRuns} run{'s' if nRuns != 1 else ''})"
)
plt.legend()
plt.tight_layout()

#### Direct vs. indirect curvature estimation error (Fig 1g)
Compare curvature-estimation error for the direct and AR(1)-based indirect methods under noisy conditions. Error is measured relative to the mean noiseless curvature; lines show the mean across simulations and shaded regions show the central 95%.

In [ ]:
# Direct estimation error
dir_error = noisy_est_curv - noiseless_mean_curv
dir_mean = dir_error.mean(axis=0)
dir_lower = np.percentile(dir_error, 2.5, axis=0)
dir_upper = np.percentile(dir_error, 97.5, axis=0)

# Indirect estimation error
ind_error = rec_noisy_est_curv - noiseless_mean_curv
ind_mean = ind_error.mean(axis=0)
ind_lower = np.percentile(ind_error, 2.5, axis=0)
ind_upper = np.percentile(ind_error, 97.5, axis=0)

# Plot
plt.figure(figsize=(5, 5))

plt.axhline(0, linestyle="--", color="black")

plt.plot(noiseless_mean_curv, dir_mean, "o-", color="grey",
         markersize=5, label="Direct")
plt.fill_between(noiseless_mean_curv, dir_lower, dir_upper,
                 color="grey", alpha=0.1, edgecolor="none")

plt.plot(noiseless_mean_curv, ind_mean, "o-", color="blue",
         markersize=5, label="Indirect")
plt.fill_between(noiseless_mean_curv, ind_lower, ind_upper,
                 color="blue", alpha=0.1, edgecolor="none")

plt.xlabel("Ground-truth curvature")
plt.ylabel("Curvature estimation error")
plt.title(f"Direct vs. indirect curvature estimation ({nRuns} run{'s' if nRuns != 1 else ''})")
plt.legend()
plt.tight_layout()
plt.show()